# Energy data review

Quick visual review of the **inputs and built outputs** of the energy pipeline.

This is a *developer* tool: it only reads the standard files the Snakemake pipeline writes (GeoParquet, PyPSA NetCDF, validation JSON). It is not part of any rule, and the packaged `energy` model stays visualisation-free — production rendering lives in the separate viewer (nismod/irv-standalone).

Build products first (see `../01-build-network`), then run top-to-bottom.

In [ ]:
import pathlib
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
import _helpers as h  # dev-only: reads pipeline outputs, not the energy package

print("repo root:", h.REPO_ROOT)
print("built products:", h.available_products())

## Which products are built?

In [ ]:
import pandas as pd

pd.DataFrame([h.summarise(p) for p in h.available_products()])[
    ["product", "nodes", "edges", "crs"]
]

## Map each built product

In [ ]:
import matplotlib.pyplot as plt

built = h.available_products()
n = max(len(built), 1)
fig, axes = plt.subplots(1, n, figsize=(7 * n, 9))
axes = [axes] if n == 1 else list(axes)
for ax, name in zip(axes, built):
    h.plot_network(name, ax=ax)
plt.tight_layout()

## Validation report

Each product writes a `validation.json` under `data/out/energy/<product>/` with sanity checks (counts, capacity/length bounds). Inspect one:

In [ ]:
name = h.available_products()[0]
h.load_validation(name)

## Nightlight raster (optional input)

The inferred products retain OSM roads near VIIRS nightlight targets. If the composite raster is present, preview it:

In [ ]:
import rioxarray

tif = h.DATA_ROOT / "incoming/energy/nightlights/viirs-mauritius-rodrigues-2024.tif"
if tif.exists():
    da = rioxarray.open_rasterio(tif, masked=True).squeeze()
    da.plot.imshow(robust=True, figsize=(9, 7))
else:
    print("nightlight composite not found:", tif)